In [3]:
import requests
import pandas as pd
import time
import urllib.parse
import json
from datetime import datetime

# ==============================================================================
# 1. CONFIGURATION DU SCRAPER ET DES 5 LIGUES
# ==============================================================================
SCRAPER_API_KEY = '7f89125c440bc88d18a73169651699d0'  # clé de notre ScraperAPI

# Dictionnaire contenant les 5 championnats à scraper
LEAGUES_CONFIG = {
    "Premier League": {"tournament_id": 17, "season_id": 76986, "url_slug": "premier-league/17"},
    "Serie A":        {"tournament_id": 23, "season_id": 76457, "url_slug": "serie-a/23"},
    "LaLiga":         {"tournament_id": 8,  "season_id": 77559, "url_slug": "laliga/8"},
    "Bundesliga":     {"tournament_id": 35, "season_id": 77333, "url_slug": "bundesliga/35"},
    "Ligue 1":        {"tournament_id": 34, "season_id": 77356, "url_slug": "ligue-1/34"}
}

def get_scraped_url(target_url, render=False):
    params = {
        'api_key': SCRAPER_API_KEY,
        'url': target_url,
    }
    if render:
        params['render'] = 'true'
    return f"http://api.scraperapi.com?{urllib.parse.urlencode(params)}"

def trouver_cle(data, target_key):
    """Parcourt récursivement le JSON pour trouver une clé spécifique."""
    if isinstance(data, dict):
        for k, v in data.items():
            if k.lower() == target_key.lower():
                return v
            if isinstance(v, (dict, list)):
                item = trouver_cle(v, target_key)
                if item is not None:
                    return item
    elif isinstance(data, list):
        for element in data:
            item = trouver_cle(element, target_key)
            if item is not None:
                return item
    return None

def fetch_league_players(league_name, config):
    league_players = []
    page = 0
    tournament_id = config["tournament_id"]
    season_id = config["season_id"]
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Origin": "https://www.sofascore.com",
        "Referer": f"https://www.sofascore.com/en-us/football/tournament/england/{config['url_slug']}"
    }

    sofascore_api_url = f"https://api.sofascore.com/api/v1/unique-tournament/{tournament_id}/season/{season_id}/statistics"

    print(f"\nDébut de la collecte : {league_name} (Saison ID {season_id})")

    while True:
        limit = 20
        offset = page * limit
        paginated_url = f"{sofascore_api_url}?limit={limit}&offset={offset}&order=-rating&accumulation=total&group=summary"
        
        proxy_url = get_scraped_url(paginated_url, render=False)
        
        try:
            print(f"[{league_name}] Extraction de la page {page + 1} (offset {offset})...")
            response = requests.get(proxy_url, headers=headers, timeout=45)
            
            if response.status_code != 200:
                print(f" Erreur HTTP {response.status_code}. Arrêt pour cette ligue.")
                break
                
            data = response.json()
            results = data.get('results', [])
            
            if not results:
                print(f" Fin des données pour la {league_name}.")
                break

            for entry in results:
                player = entry.get('player', {})
                team = entry.get('team', {})
                
                # Récupération dynamique
                nom = trouver_cle(entry, 'name') or trouver_cle(entry, 'shortName') or 'Inconnu'
                equipe = trouver_cle(team, 'name') or 'Sans Équipe'
                
                buts = trouver_cle(entry, 'goals') or 0
                assists = trouver_cle(entry, 'assists') or 0
                xg = trouver_cle(entry, 'expectedGoals') or 0.0
                tacles = trouver_cle(entry, 'tackles') or trouver_cle(entry, 'tacklesWon') or 0
                
                # Correction clé dribble : 'successfulDribbles' est le vrai nom de variable dans l'API
                succ_dribbles = trouver_cle(entry, 'successfulDribbles') or trouver_cle(entry, 'dribbles') or 0
                
                total_passes = trouver_cle(entry, 'totalPasses') or 0
                passes_reussies = trouver_cle(entry, 'accuratePasses') or 0
                passes_pct = trouver_cle(entry, 'accuratePassesPercentage')
                if passes_pct is None:
                    passes_pct = round((passes_reussies / total_passes) * 100, 2) if total_passes > 0 else 0.0

                player_data = {
                    'Ligue': league_name,
                    'Equipe': equipe,
                    'Nom': nom,
                    'Buts': int(buts),
                    'xG': round(float(xg), 2),
                    'Succ_dribbles': int(succ_dribbles),  # Corrigé : pointe vers la bonne variable
                    'Tacles': int(tacles),
                    'Assists': int(assists),
                    'Passes_Reussies_Pct': round(float(passes_pct), 2),
                }
                league_players.append(player_data)
                
            total_pages = data.get('pages', 27)
            
            if page + 1 >= total_pages:
                break
                
            page += 1
            time.sleep(1.5)  # Pause légère entre les pages pour ne pas surcharger api
            
        except Exception as e:
            print(f" Erreur rencontrée à la page {page} : {e}")
            break
            
    return league_players

# ==============================================================================
# 2. BOUCLE PRINCIPALE SUR LES 5 LIGUES ET SAUVEGARDE UNIQUE
# ==============================================================================
all_leagues_data = []

for league, config in LEAGUES_CONFIG.items():
    data_ligue = fetch_league_players(league, config)
    all_leagues_data.extend(data_ligue)
    print(f" Collecté {len(data_ligue)} joueurs pour la {league}.\n" + "-"*40)

# Exportation globale
if all_leagues_data:
    df_global = pd.DataFrame(all_leagues_data)
    
    # Réorganisation des colonnes dans l'ordre strict demandé
    colonnes_ordonnees = ['Ligue', 'Equipe', 'Nom', 'Buts', 'xG', 'Succ_dribbles', 'Tacles', 'Assists', 'Passes_Reussies_Pct']
    df_global = df_global[colonnes_ordonnees]
    
    output_file = "sofascore_top5leagues_players_stat.csv"
    
    try:
        df_global.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n [Succès] Total de {len(df_global)} joueurs collectés sur les 5 championnats.")
        print(f" Fichier unique exporté avec succès : '{output_file}'")
    except PermissionError:
        alt_file = f"sofascore_top5_players_{int(time.time())}.csv"
        df_global.to_csv(alt_file, index=False, encoding='utf-8-sig')
        print(f"\n Fichier d'origine verrouillé. Données écrites à la place dans '{alt_file}' !")
else:
    print("\n Aucune donnée n'a pu être extraite des championnats.")


Début de la collecte : Premier League (Saison ID 76986)
[Premier League] Extraction de la page 1 (offset 0)...
[Premier League] Extraction de la page 2 (offset 20)...
[Premier League] Extraction de la page 3 (offset 40)...
[Premier League] Extraction de la page 4 (offset 60)...
[Premier League] Extraction de la page 5 (offset 80)...
[Premier League] Extraction de la page 6 (offset 100)...
[Premier League] Extraction de la page 7 (offset 120)...
[Premier League] Extraction de la page 8 (offset 140)...
[Premier League] Extraction de la page 9 (offset 160)...
[Premier League] Extraction de la page 10 (offset 180)...
[Premier League] Extraction de la page 11 (offset 200)...
[Premier League] Extraction de la page 12 (offset 220)...
[Premier League] Extraction de la page 13 (offset 240)...
[Premier League] Extraction de la page 14 (offset 260)...
[Premier League] Extraction de la page 15 (offset 280)...
[Premier League] Extraction de la page 16 (offset 300)...
[Premier League] Extraction de

In [2]:
import pandas as pd

df = pd.read_csv("sofascore_top5leagues_players_stat.csv")
df

,Ligue,Equipe,Nom,Buts,xG,Succ_dribbles,Tacles,Assists,Passes_Reussies_Pct
0,Premier League,Manchester United,Bruno Fernandes,9,10.83,15,54,21,82.20
1,Premier League,Burnley,Max Weiss,0,0.00,0,0,0,60.71
2,Premier League,Arsenal,Declan Rice,4,3.17,14,70,5,87.31
3,Premier League,Newcastle United,Bruno Guimarães,9,5.60,18,62,5,86.15
4,Premier League,Manchester City,Rodri,1,1.13,7,41,0,90.11
...,...,...,...,...,...,...,...,...,...
2768,Ligue 1,Stade Rennais,Henrick Do Marcolino,0,0.00,0,1,0,33.33
2769,Ligue 1,Le Havre,Georges Gomis,0,0.00,0,0,0,0.00
2770,Ligue 1,Auxerre,Yvan Zaddy,0,0.00,0,0,0,100.00
2771,Ligue 1,Stade Rennais,Lucas Rosier,0,0.00,0,0,0,0.00
